# Lexicon corpus - inspect & edit

Durable, package-level notebook for interacting with the Parquet corpus under
`data/lexicon/` (distinct from `scratch_space/`, which holds throwaway phase
work). It is a **thin caller**: every operation is a package function; no logic
lives here.

- `LexiconStore.from_data_fol` - build the read/query store over a corpus folder.
- `inspect_table` - read-only DuckDB SQL over the Parquet (QA / exploration).
- `export_table` / `import_table` - the validated edit round-trip.

Requires the `store` extra for the DuckDB inspect path (`uv sync --extra store`).
The corpus is the full one from the ingestion phase, or the sample produced from
the committed seed by `parquetize_seed.ipynb`. **A fresh checkout has no corpus:**
run `parquetize_seed.ipynb` first, or `from_data_fol` raises `CorpusNotFoundError`.


In [ ]:
from lang_tools.lexicon.corpus import export_table
from lang_tools.lexicon.corpus import import_table
from lang_tools.lexicon.corpus import inspect_table
from lang_tools.lexicon.lemma_store import LexiconStore
from lang_tools.params.lang_tools_params import get_lang_tools_params

data_fol = get_lang_tools_params().paths.data_fol
# data_fol = Path.home() / "some" / "other" / "corpus"  # swap to inspect a different corpus folder
data_fol

## Build a store over the corpus

`from_data_fol` reads `<data_fol>/lexicon/` Parquet into the SQLite engine and
raises `CorpusNotFoundError` when the folder holds no corpus yet.


In [ ]:
store = LexiconStore.from_data_fol(data_fol)
store.get_all_concepts()[:3]

## Inspect (read-only)

Run SQL over a table's Parquet without an import step. `where` / `limit` are
optional; `lang` reads one language file of a partitioned table.


In [ ]:
inspect_table("concepts", data_fol=data_fol, limit=10)

In [ ]:
inspect_table("lemmas", data_fol=data_fol, lang="pt", where="language = 'pt'", limit=10)

In [ ]:
inspect_table("senses", data_fol=data_fol, lang="pt", limit=10)

## Edit round-trip (validated)

Export a table to a transient JSONL, hand/LLM-edit it, then re-import. The
import validates every row through the pydantic model before rewriting the
canonical Parquet, so a renamed/missing column or bad value fails loudly. The
JSONL is scratch - never commit it. Schema changes are not edits: regenerate
from the ingestion pipeline instead.


In [ ]:
scratch = data_fol / "_scratch" / "concepts.jsonl"
export_table("concepts", scratch, data_fol=data_fol)
# ... edit `scratch` by hand or with an LLM, then re-import ...
# import_table("concepts", scratch, data_fol=data_fol)